# 01 — Data Collection: SpaceX REST API

Pulls Falcon 9 launch, rocket, payload, and launchpad data from the public
SpaceX API (https://api.spacexdata.com/v4) and assembles the raw launch
DataFrame used by the rest of this project.

**Requires internet access to run** — open this in Google Colab or a local
Jupyter environment with a network connection.

In [ ]:
import requests
import pandas as pd

BASE = "https://api.spacexdata.com/v4"


### Fetch launches, rockets, payloads, and launchpads

In [ ]:
launches = requests.get(f"{BASE}/launches/past").json()
rockets = {r["id"]: r for r in requests.get(f"{BASE}/rockets").json()}
launchpads = {p["id"]: p for p in requests.get(f"{BASE}/launchpads").json()}
payloads = {p["id"]: p for p in requests.get(f"{BASE}/payloads").json()}

print(f"Fetched {len(launches)} launch records")


### Build the raw launch table

In [ ]:
rows = []
for l in launches:
    rocket = rockets.get(l.get("rocket"), {})
    pad = launchpads.get(l.get("launchpad"), {})
    payload_ids = l.get("payloads", [])
    payload_mass = None
    orbit = None
    if payload_ids:
        p = payloads.get(payload_ids[0], {})
        payload_mass = p.get("mass_kg")
        orbit = p.get("orbit")

    cores = l.get("cores", [{}])
    core = cores[0] if cores else {}

    rows.append({
        "FlightNumber": l.get("flight_number"),
        "Date": l.get("date_utc"),
        "BoosterVersion": rocket.get("name"),
        "PayloadMass": payload_mass,
        "Orbit": orbit,
        "LaunchSite": pad.get("name"),
        "Outcome": core.get("landing_success"),
        "LandingType": core.get("landing_type"),
        "Reused": core.get("reused"),
        "Legs": core.get("legs"),
        "GridFins": core.get("gridfins"),
        "Block": core.get("block"),
        "Latitude": pad.get("latitude"),
        "Longitude": pad.get("longitude"),
    })

df = pd.DataFrame(rows)
df.head()


### Filter to Falcon 9 (drop Falcon 1) and save

In [ ]:
df_f9 = df[df["BoosterVersion"] == "Falcon 9"].reset_index(drop=True)
print(df_f9.shape)
df_f9.to_csv("../falcon9_api_raw.csv", index=False)
df_f9.head()
